In [6]:
from pathlib import Path
from glob import glob
import pandas as pd
import numpy as np
import psutil
import gc
import os
from joblib import Parallel, delayed
from tqdm import tqdm

from wtpcc.pclib import pcf

ModuleNotFoundError: No module named 'wtpcc'

In [2]:
data = Path("../data/")

In [10]:
files = data.glob("*.csv")
print(list(files)[0])


..\data\WT_ID_10_selected_signals_imputed.csv


In [5]:
pc = Path("../data/Powercurve/powercurve.csv")
pc_df = pd.read_csv(pc)
display(pc_df)


,Wind speed (m/s),Power (kW)
0,4.0,55.0
1,4.5,110.0
2,5.0,186.0
3,5.5,264.0
4,6.0,342.0
5,6.5,424.0
6,7.0,506.0
7,7.5,618.0
8,8.0,730.0
9,8.5,865.0


In [26]:
def pc_filtering(
    scadaData: pd.DataFrame,
    powerCurve: pd.DataFrame,
    windSpec: str,
    powerSpec: str,
    windowSize: int,
    powerMargin: float = 150.0,
    minWindSpeed: float = 5.0,
    measureRAM: bool = False,
) -> tuple[pd.DataFrame, int | None]:
    
    windVals = scadaData[windSpec].to_numpy()
    powerVals = scadaData[powerSpec].to_numpy()
    
    pcWind = powerCurve[windSpec].to_numpy()
    pcPower = powerCurve[powerSpec].to_numpy()
    
    pc = np.interp(x=windVals,
                   xp=pcWind,
                   fp=pcPower)
    
    pcRange = ( windVals >= pcWind[0]) & (windVals <= pcWind[-1])
    
    minWind = windVals > minWindSpeed
    
    outOfBand = ((np.abs(powerVals - pc) > powerMargin) & pcRange & minWind).astype(np.int8)
    
    _filter = np.ones(windowSize, dtype=int)
    
    transitions = np.convolve(outOfBand, _filter, mode="same")
    
    mask = pcRange & minWind & (transitions == 0)
    
    filteredData = scadaData.loc[mask].copy()
    
    if measureRAM:
        ramUsage = psutil.virtual_memory().available
        return filteredData, ramUsage
    
    return filteredData, None

def _read_file(path: Path) -> pd.DataFrame:
    _format = path.suffix.lower()
    
    if _format == ".csv":
        return pd.read_csv(path)
    if _format == ".parquet":
        return pd.read_parquet(path)
    
    raise ValueError(f"Unknown file type: {path}")

def _write_file(df: pd.DataFrame, path: Path) -> None:
    _format = path.suffix.lower()
    csv = ".csv"
    parquet = ".parquet"

    if _format == csv:
        return df.to_csv(path, index=False)
    if _format == parquet:
        return df.to_parquet(path, index=False)
    
    raise ValueError(f"Unknown file type: {path}"
                     f"Known file types {[csv,parquet]}.")

def _process_file(
    filePath: Path,
    outDir: Path,
    powerCurve: pd.DataFrame,
    windSpec: str,
    powerSpec: str,
    windowSize: int,
    powerMargin: float = 150.0,
    minWindSpeed: float = 5.0,
    measureRAM: bool = False,
) -> int | None:
    
    scadaData = _read_file(filePath)
    
    filteredData, ram = pc_filtering(
        scadaData=scadaData,
        powerCurve=powerCurve,
        windSpec=windSpec,
        powerSpec=powerSpec,
        windowSize=windowSize,
        powerMargin=powerMargin,
        minWindSpeed=minWindSpeed,
        measureRAM=measureRAM,
    )
    
    _write_file(filteredData, outDir / ("pc_filtered_"+filePath.name))
    
    del scadaData
    del filteredData
    gc.collect()
    
    return ram

def _estimate_jobs(ramBefore: int, ramAfter: int) -> int:
    cpuCnt = max((os.cpu_count() or 1) -1, 1)
    
    ramUsed = max(ramBefore - ramAfter, 1)
    avail_ram = psutil.virtual_memory().available
    
    jobs = max(1, avail_ram // ramUsed)
    
    return max(1, min(cpuCnt, jobs))


def pc(
    inputDir: str,
    outDir: str,
    powerCurve: pd.DataFrame,
    windSpec: str,
    powerSpec: str,
    windowSize: int,
    powerMargin: float = 150,
    minWindSpeed: float = 5.0,
    nJobs: int | None = None,
) -> None:
    
    inputDir = Path(inputDir)
    outDir = Path(outDir)
    outDir.mkdir(parents=True, exist_ok=True)
    
    files = list(inputDir.glob("*.*"))
    
    if not files:
        raise ValueError(f"No files found  in {inputDir}")
    
    if nJobs is None:
        ramBefore = psutil.virtual_memory().available
    
        ramAfter = _process_file(
            filePath=files[0],
            outDir=outDir,
            powerCurve=powerCurve,
            windSpec=windSpec,
            powerSpec=powerSpec,
            windowSize=windowSize,
            powerMargin=powerMargin,
            minWindSpeed=minWindSpeed,
            measureRAM=True
        )
        if len(files) == 1:
            return
        
        jobs = _estimate_jobs(ramBefore, ramAfter)
        
    Parallel(n_jobs=nJobs if nJobs is not None else jobs, backend="loky", verbose=10)(
        delayed(_process_file)(
            filePath=fp,
            outDir=outDir,
            powerCurve=powerCurve,
            windSpec=windSpec,
            powerSpec=powerSpec,
            windowSize=windowSize,
            powerMargin=powerMargin,
            minWindSpeed=minWindSpeed,
        )
        for fp in tqdm(files[1:] if nJobs is None else files)
    )

    

In [3]:
input = Path("../data/")
pc_path = Path("../data/powercurve/powercurve.csv")



In [33]:
pc(
    inputDir=input,
    powerCurve= pd.read_csv(pc_path),
    outDir=Path("../data/pc_filtered/"),
    windSpec="Wind speed (m/s)",
    powerSpec="Power (kW)",
    windowSize=60,
)

100%|██████████| 13/13 [00:00<00:00, 796.23it/s]
[Parallel(n_jobs=15)]: Done   2 out of  13 | elapsed:   37.8s remaining:  3.5min
[Parallel(n_jobs=15)]: Done   4 out of  13 | elapsed:   39.3s remaining:  1.5min
[Parallel(n_jobs=15)]: Done   6 out of  13 | elapsed:   40.3s remaining:   47.0s
[Parallel(n_jobs=15)]: Done   8 out of  13 | elapsed:   40.3s remaining:   25.2s
[Parallel(n_jobs=15)]: Done  10 out of  13 | elapsed:   40.6s remaining:   12.1s
[Parallel(n_jobs=15)]: Done  13 out of  13 | elapsed:   41.0s finished


In [34]:
pc_f = Path("../data/pc_filtered/")
files = list(pc_f.glob("*.csv"))

df = pd.read_csv(files[0])
display(df)

,WT_ID,Ambient temperature (converter) (°C),Date and time,Drive train acceleration (mm/ss),Gear oil inlet pressure (bar),Gear oil pump pressure (bar),Gearbox speed (RPM),Generator bearing front temperature (°C),Generator bearing rear temperature (°C),Generator RPM (RPM),...,Blade angle (pitch position) C (°),Front bearing temperature (°C),Gear oil inlet temperature (°C),Gear oil temperature (°C),Rear bearing temperature (°C),Tower Acceleration X (mm/ss),Tower Acceleration Y (mm/ss),Transformer cell temperature (°C),Transformer temperature (°C),Yaw bearing angle (°)
0,10,11.012117,2016-06-28 00:00:00,4.847309,103.217178,317.149118,1243.851321,39.128745,38.606996,1243.218933,...,1.581904,60.327041,47.700036,53.496310,57.507342,57.565641,21.071799,13.695659,43.373982,206.883805
1,10,11.012117,2016-06-28 00:10:00,4.847309,103.217178,317.149118,1243.851321,39.128745,38.606996,1243.218933,...,1.581904,60.327041,47.700036,53.496310,57.507342,57.565641,21.071799,13.695659,43.373982,206.883805
2,10,11.012117,2016-06-28 00:20:00,4.847309,103.217178,317.149118,1243.851321,39.128745,38.606996,1243.218933,...,1.581904,60.327041,47.700036,53.496310,57.507342,57.565641,21.071799,13.695659,43.373982,206.883805
3,10,11.012117,2016-06-28 00:30:00,4.847309,103.217178,317.149118,1243.851321,39.128745,38.606996,1243.218933,...,1.581904,60.327041,47.700036,53.496310,57.507342,57.565641,21.071799,13.695659,43.373982,206.883805
4,10,11.012117,2016-06-28 00:40:00,4.847309,103.217178,317.149118,1243.851321,39.128745,38.606996,1243.218933,...,1.581904,60.327041,47.700036,53.496310,57.507342,57.565641,21.071799,13.695659,43.373982,206.883805
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79977,10,5.933333,2022-12-31 23:00:00,8.067195,184.505280,594.568803,1192.547016,41.986666,37.108333,1191.377003,...,0.793333,53.253334,44.075000,48.156667,51.095000,84.934917,21.212455,8.250000,32.321667,54.536945
79978,10,5.803333,2022-12-31 23:20:00,8.475813,183.057751,585.725451,1215.557022,41.308334,37.271666,1214.589432,...,0.707000,53.858333,46.436206,48.595001,51.770000,91.181917,25.429733,7.951667,32.373333,54.536945
79979,10,5.750000,2022-12-31 23:30:00,6.882125,187.218025,594.446427,1247.046646,38.722413,35.406896,1245.672648,...,0.153103,54.536208,45.866667,48.925863,53.008621,83.474661,24.053980,7.858621,32.415516,54.536945
79980,10,5.895000,2022-12-31 23:40:00,7.076928,191.511204,602.850081,1297.614861,38.223334,35.140000,1296.801697,...,0.281000,55.155001,45.749999,49.173333,53.718333,81.013649,23.700713,7.936667,32.378333,58.633933


In [4]:
pcf.pc(
    inputDir=input,
    powerCurve= pd.read_csv(pc_path),
    outDir=Path("../data/pc_filtered/"),
    windSpec="Wind speed (m/s)",
    powerSpec="Power (kW)",
    windowSize=60,
)

NameError: name 'pcf' is not defined